# Phase 3: 1000 km Autonomous Integrated Air & Missile Defense (IAMD) Simulation

**Author:** Autonomous Aerospace Simulation Engineer & Data Scientist  
**Scope:** 1000 km Dual-Zone Operational Matrix, 3D True Proportional Navigation (TPN), 1,000-Run Monte Carlo Simulation, Interactive Plotly Visualizer, and Machine Learning Threat Trajectory Predictor.

---

## Executive Summary
This standalone simulation implements a comprehensive, end-to-end Integrated Air and Missile Defense (IAMD) architecture across a 1,000 km theater:
1. **Kinematics & Threat Profiles**: High-Flying Ballistic, Quasi-Ballistic with lateral weave, and Supersonic Cruise threats paired with a 30G-capable True Proportional Navigation (TPN) interceptor.
2. **Geospatial Battlefield & Resource Allocation**: 10 Aggressor launch sites ($X \in [0, 100]\text{ km}$) and 10 Defender battery sites ($X \in [900, 1000]\text{ km}$) with early warning radar trajectory projection and $t_{\text{go}}$ minimization.
3. **Monte Carlo Engine**: 1,000-run engagement batch logging telemetry, CPA miss distance, and outcomes to structured CSV.
4. **Interactive 3D Visualizer**: Plotly 3D rendering with detonation bursts, CPA miss vectors, and trajectory animation.
5. **Machine Learning Model**: Multi-task classification (optimal defender site dispatch) and regression (predicted intercept coordinates and time-to-go) with sub-millisecond inference.


## Milestone 1: Open-Source Performance Benchmarks & Kinematics Setup

### 1.1 Mathematical Formulation of Kinematics & Coordinate Frame
The battlefield operates in a right-handed 3D Cartesian frame:
- $X$ **(Downrange, meters)**: Longitudinal separation axis from Aggressor Zone ($X=0$) to Defender Zone ($X=1,000\text{ km}$).
- $Y$ **(Crossrange, meters)**: Lateral span ($Y \in [-100, 100]\text{ km}$).
- $Z$ **(Altitude, meters)**: Vertical axis ($Z \ge 0$).

#### Environmental Models (Gravity & Atmospheric Density)
$$\vec{g} = \begin{bmatrix} 0 \\ 0 \\ -9.81 \end{bmatrix} \text{ m/s}^2, \quad \rho(z) = \rho_0 e^{-z / H} \quad (\rho_0 = 1.225\text{ kg/m}^3, H = 7500\text{ m})$$

### 1.2 Open-Source Threat & Interceptor Profiles
1. **High-Flying Ballistic Threat**:
   - Velocity: Mach 5.5–7.5 ($1,870 - 2,550\text{ m/s}$)
   - Trajectory: Exo-atmospheric parabolic apogee $> 80\text{ km}$ ($80 - 140\text{ km}$) with gravity acceleration and lower atmosphere terminal reentry drag (ballistic coefficient $\beta \approx 9,000\text{ kg/m}^2$).
2. **Quasi-Ballistic Threat**:
   - Velocity: Mach 4.5–6.0 ($1,530 - 2,040\text{ m/s}$)
   - Trajectory: Depressed boost-glide with midcourse pull-up ($30 - 45\text{ km}$ altitude), periodic lateral weave acceleration $a_{\text{lat}} = A \sin(\omega t)$ ($A \in [25, 45]\text{ m/s}^2, \omega \in [0.10, 0.20]\text{ rad/s}$), and terminal dive.
3. **Supersonic Cruise Threat**:
   - Velocity: Mach 2.5–3.5 ($850 - 1,190\text{ m/s}$)
   - Trajectory: Low-altitude terrain-following profile ($Z \in [1.5, 4.5]\text{ km}$) with terminal evasive maneuverability ($a_{\text{evade}} \in [30, 50]\text{ m/s}^2$).
4. **Defender Interceptor**:
   - Velocity: Boosted Mach 5.0–8.0 ($1,700 - 2,720\text{ m/s}$)
   - Maneuvering Ceiling: $30\text{G}$ ($a_{\text{max}} = 30 \times 9.81 = 294.3\text{ m/s}^2$)
   - 3D True Proportional Navigation (TPN) Guidance Law:
     $$\vec{r}_{\text{rel}} = \vec{r}_T - \vec{r}_I, \quad \vec{v}_{\text{rel}} = \vec{v}_T - \vec{v}_I$$
     $$\vec{\Omega} = \frac{\vec{r}_{\text{rel}} \times \vec{v}_{\text{rel}}}{\|\vec{r}_{\text{rel}}\|^2}, \quad \vec{a}_{\text{cmd}} = N \cdot \|\vec{v}_I\| \cdot (\vec{\Omega} \times \hat{r}_{\text{rel}})$$
     Clamped to $\|\vec{a}_{\text{cmd}}\| \le a_{\text{max}}$.

### 1.3 Detonation Constraints
- **Sub-timestep Closest Point of Approach (CPA)**:
  $$t_{\text{cpa}} = -\frac{\vec{r}_{\text{rel}} \cdot \vec{v}_{\text{rel}}}{\|\vec{v}_{\text{rel}}\|^2}$$
  $$\vec{p}_{T, \text{cpa}} = \vec{r}_T + \vec{v}_T t_{\text{cpa}}, \quad \vec{p}_{I, \text{cpa}} = \vec{r}_I + \vec{v}_I t_{\text{cpa}}$$
  $$d_{\text{cpa}} = \|\vec{p}_{T, \text{cpa}} - \vec{p}_{I, \text{cpa}}\|$$
- **Hit / Miss Criterion**:
  - $\text{CPA} \le 15.0\text{ m} \implies \mathbf{HIT}$ (Proximity fuse lethal trigger & kill)
  - $\text{CPA} > 15.0\text{ m} \implies \mathbf{MISS}$ (Record CPA miss distance)


In [1]:
# ==============================================================================
# 1. CORE IMPORTS & GLOBAL CONFIGURATION
# ==============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from numba import njit
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score, mean_squared_error
import time
import os

# Set random seed for scientific reproducibility
np.random.seed(42)

print("=" * 70)
print("PHASE 3: 1000 KM IAMD SIMULATION - CORE MODULES INITIALIZED")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")
print("=" * 70)


PHASE 3: 1000 KM IAMD SIMULATION - CORE MODULES INITIALIZED
NumPy: 1.26.4 | Pandas: 1.5.3


In [2]:
# ==============================================================================
# 2. OPEN-SOURCE KINEMATIC SPECIFICATIONS & PARAMETER DICTIONARY
# ==============================================================================

KINEMATIC_PROFILES = {
    'high_ballistic': {
        'name': 'High-Flying Ballistic Threat',
        'mach_range': (5.5, 7.5),
        'speed_range_ms': (1870.0, 2550.0),
        'apogee_range_m': (80000.0, 140000.0),
        'ballistic_coeff_beta': 9000.0,
        'description': 'Exo-atmospheric parabolic trajectory with terminal atmospheric reentry drag'
    },
    'quasi_ballistic': {
        'name': 'Quasi-Ballistic Maneuvering Threat',
        'mach_range': (4.5, 6.0),
        'speed_range_ms': (1530.0, 2040.0),
        'cruise_alt_range_m': (30000.0, 45000.0),
        'weave_amp_range_ms2': (25.0, 45.0),
        'weave_freq_range_rads': (0.10, 0.20),
        'description': 'Depressed boost-glide with midcourse pull-up, periodic lateral weave, and terminal dive'
    },
    'supersonic_cruise': {
        'name': 'Supersonic Low-Altitude Cruise Threat',
        'mach_range': (2.5, 3.5),
        'speed_range_ms': (850.0, 1190.0),
        'cruise_alt_range_m': (1500.0, 4500.0),
        'terminal_evasion_accel_ms2': (30.0, 50.0),
        'description': 'Terrain-following low-altitude cruise with high-G terminal evasive maneuvers'
    },
    'interceptor': {
        'name': 'Defender Kinetic Interceptor',
        'mach_range': (5.0, 8.0),
        'speed_range_ms': (1700.0, 2720.0),
        'max_g_ceiling': 30.0,
        'max_accel_ms2': 30.0 * 9.81,
        'nav_gain_N': 4.5,
        'lethal_radius_m': 15.0,
        'description': 'High-velocity 3D True Proportional Navigation (TPN) interceptor with 30G ceiling'
    }
}

df_specs = pd.DataFrame([
    {
        'Profile': v['name'],
        'Mach Range': f"Mach {v['mach_range'][0]:.1f} - {v['mach_range'][1]:.1f}",
        'Velocity (m/s)': f"{v['speed_range_ms'][0]:.0f} - {v['speed_range_ms'][1]:.0f} m/s",
        'Flight Regimes & Capabilities': v['description']
    }
    for k, v in KINEMATIC_PROFILES.items()
])

print("Kinematic Specifications Table:")
print(df_specs.to_string(index=False))


Kinematic Specifications Table:
                              Profile     Mach Range  Velocity (m/s)                                                           Flight Regimes & Capabilities
         High-Flying Ballistic Threat Mach 5.5 - 7.5 1870 - 2550 m/s             Exo-atmospheric parabolic trajectory with terminal atmospheric reentry drag
   Quasi-Ballistic Maneuvering Threat Mach 4.5 - 6.0 1530 - 2040 m/s Depressed boost-glide with midcourse pull-up, periodic lateral weave, and terminal dive
Supersonic Low-Altitude Cruise Threat Mach 2.5 - 3.5  850 - 1190 m/s            Terrain-following low-altitude cruise with high-G terminal evasive maneuvers
         Defender Kinetic Interceptor Mach 5.0 - 8.0 1700 - 2720 m/s        High-velocity 3D True Proportional Navigation (TPN) interceptor with 30G ceiling


In [3]:
# ==============================================================================
# 3. SUB-TIMESTEP CLOSEST POINT OF APPROACH (CPA) & PROXIMITY DETONATION
# ==============================================================================

def compute_substep_cpa(pos1, vel1, pos2, vel2, dt):
    """
    Computes exact continuous-time Closest Point of Approach (CPA)
    within the discrete timestep interval [0, dt].
    """
    r_rel = pos1 - pos2
    v_rel = vel1 - vel2
    v_rel_sq = np.dot(v_rel, v_rel)
    
    if v_rel_sq < 1e-8:
        return 0.0, np.linalg.norm(r_rel), pos1.copy(), pos2.copy()
        
    t_cpa = -np.dot(r_rel, v_rel) / v_rel_sq
    
    # Clamp to current sub-step [0, dt]
    t_eval = np.clip(t_cpa, 0.0, dt)
    p1_cpa = pos1 + vel1 * t_eval
    p2_cpa = pos2 + vel2 * t_eval
    miss_dist = np.linalg.norm(p1_cpa - p2_cpa)
    
    return t_eval, miss_dist, p1_cpa, p2_cpa

# Unit Test CPA calculation
p_threat_test = np.array([500000.0, 0.0, 25000.0])
v_threat_test = np.array([1800.0, 0.0, 0.0])
p_int_test = np.array([500090.0, 8.0, 25000.0])
v_int_test = np.array([-2200.0, 0.0, 0.0])

t_cpa_test, miss_test, p1_c, p2_c = compute_substep_cpa(p_threat_test, v_threat_test, p_int_test, v_int_test, dt=0.05)
outcome_test = "HIT" if miss_test <= 15.0 else "MISS"

print(f"CPA Sub-Step Unit Test:")
print(f"  t_cpa within step: {t_cpa_test*1000:.3f} ms | Miss Distance: {miss_test:.2f} m | Outcome: {outcome_test}")
assert abs(miss_test - 8.0) < 1e-4, "CPA math verification failed"
print("  => Mathematical formulation and detonation criteria verified successfully!")


CPA Sub-Step Unit Test:
  t_cpa within step: 22.500 ms | Miss Distance: 8.00 m | Outcome: HIT
  => Mathematical formulation and detonation criteria verified successfully!
